In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,END
from langgraph.graph.message import add_messages
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage
from typing import TypedDict,Annotated

In [2]:
load_dotenv()

True

In [3]:
class MyState(TypedDict):
    draft:str
    approval:str
    review_comments : str
    messages: Annotated[list,add_messages]

In [4]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [5]:
@tool
def send_email(draft:str)->dict:
    """This tool sends email"""
    print(draft)
    print("Email sent!")



In [6]:
llm_with_tools  = llm.bind_tools([send_email])

In [22]:
def llm_node(state:MyState)->dict:
    system = SystemMessage(content="You are a Helpful assitent.")
    response = llm_with_tools.invoke([system]+state["messages"])
    return {'messages':[response]}

def generate_email_draft(state:MyState)->dict:
    mes = state['messages']
    print("This is state:",state)
    if state.get('approval') is None:
        draft = llm_with_tools.invoke(mes)
    else:
        comments = HumanMessage(content=state['review_comments'])
        draft = llm_with_tools.invoke([comments]+mes)
    return {'draft':draft.content}


send_email_node = ToolNode([send_email])

def review_email(state:MyState)->dict:

    if state.get('draft') is not None:
        print("Review the email and provide approval (yes/no/edit) at the end.")
        print(state['draft'])
        res = input("Approval status:")

        if res.lower() == 'yes':
            return {'approval':'yes'}
        elif res.lower() == 'no':
            return {'approval':'no'}
        else:
            comments = input("Provide review comments")
            return{'approval':'edit','review_comments':comments}
    else:
        print('review called before drafting.')
        return {}
        



In [23]:
# define routing function
def after_review(state:MyState):
    last = state["messages"][-1]

    if getattr(last,"tool_calls") and state['approval'] == 'yes':
        return "send_email"
    elif state['approval'] == 'edit':
        return "draft"
    return "end"


In [24]:
flow = StateGraph(MyState)
flow.add_node("llm_node",llm_node)
flow.add_node("draft",generate_email_draft)
flow.add_node("review",review_email)
flow.add_node("send",send_email_node)

flow.set_entry_point("llm_node")

In [25]:
flow.add_edge("llm_node","draft")
flow.add_edge("draft","review")
flow.add_edge("review","send")
flow.add_conditional_edges("review",
                           after_review,
                           {
                               'send_email':'send',
                               'draft':'draft',
                               'end':END
                           }
                           )

In [26]:
app = flow.compile()

In [28]:
result = app.invoke({'messages':[HumanMessage(content="write a very short email to my friend NaraLion.")]})

This is state: {'messages': [HumanMessage(content='write a very short email to my friend NaraLion.', additional_kwargs={}, response_metadata={}, id='e6bc6d9d-1feb-4acd-99e4-48efd77cf8f5'), AIMessage(content='Subject: Quick Hello  \n\nHey NaraLion,  \n\nJust wanted to drop a quick note to say hi and see how you’re doing. Hope all’s well!  \n\nTalk soon,  \n[Your Name]', additional_kwargs={'reasoning_content': 'We need to write a very short email to friend NaraLion. Probably we need to use send_email function? The user just says "write a very short email to my friend NaraLion." They likely want the email content, not to actually send. But we have a tool to send email. The user didn\'t provide email address or content. Probably just provide the email text. So we respond with a short email. No need to call function.'}, response_metadata={'token_usage': {'completion_tokens': 143, 'prompt_tokens': 140, 'total_tokens': 283, 'completion_time': 0.300057611, 'completion_tokens_details': {'reason

In [29]:
print(result)

{'draft': '**Subject:** Quick shout‑out  \n\nHey NaraLion,  \n\nJust saw your latest “clean‑shave” selfie—so smooth the trainhostress had to ask for a seatbelt! 😜  \n\nCatch you later,  \n[Your Name]', 'approval': 'yes', 'review_comments': 'add a savage joke about his clean shave, add words like trainhostress, similar to air hostress', 'messages': [HumanMessage(content='write a very short email to my friend NaraLion.', additional_kwargs={}, response_metadata={}, id='e6bc6d9d-1feb-4acd-99e4-48efd77cf8f5'), AIMessage(content='Subject: Quick Hello  \n\nHey NaraLion,  \n\nJust wanted to drop a quick note to say hi and see how you’re doing. Hope all’s well!  \n\nTalk soon,  \n[Your Name]', additional_kwargs={'reasoning_content': 'We need to write a very short email to friend NaraLion. Probably we need to use send_email function? The user just says "write a very short email to my friend NaraLion." They likely want the email content, not to actually send. But we have a tool to send email. The